# 🔄 Re-Training & Fine-Tuning Model 5 Kondisi Kulit (VGG16)
### Melatih Ulang `finetuned_best.h5` agar Akurasi Semakin Tinggi

Notebook ini digunakan jika Anda ingin **melatih ulang (*fine-tuning lanjutan*)** model VGG16 yang ada di SkinCheck saat ini.
Model ini tidak dilatih dari nol, melainkan memuat bobot dari `finetuned_best.h5` lalu melanjutkannya dengan dataset baru.

**Langkah Penggunaan:**
1. Buka [colab.research.google.com](https://colab.research.google.com)
2. Unggah file `finetuned_best.h5` dari folder `facial-condition-model/` ke Google Colab.
3. Jalankan sel di bawah satu per satu.

## Langkah 1: Pasang Kredensial Kaggle

In [ ]:
import os

os.environ['KAGGLE_USERNAME'] = "robimuhammadw"
os.environ['KAGGLE_KEY'] = "KGAT_9e0aed00661079fa5fe52e9f9d89ea05"

!pip install -q kaggle tensorflow
print("✅ Kredensial Kaggle siap!")

## Langkah 2: Unduh Dataset Kulit Baru dari Kaggle

In [ ]:
import zipfile

# Unduh dataset kondisi kulit wajah dari Kaggle
!kaggle datasets download -d shubhamgoel27/dermnet -p ./derm_data

zip_files = [f for f in os.listdir('./derm_data') if f.endswith('.zip')]
if zip_files:
    zip_path = os.path.join('./derm_data', zip_files[0])
    print(f"Mengekstrak {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('./dataset_retrain')
    print("✅ Ekstraksi dataset selesai!")

## Langkah 3: Muat Model Lama (`finetuned_best.h5`)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model

model_path = 'finetuned_best.h5' # Pastikan Anda sudah mengunggah file ini ke Colab
model = load_model(model_path, compile=False)

print("✅ Model VGG16 berhasil dimuat!")
print("Input:", model.input_shape)
print("Output:", model.output_shape)

## Langkah 4: Compile dengan Learning Rate Kecil & Latih Ulang

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Gunakan learning rate kecil (1e-5) agar bobot lama tidak rusak
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    './dataset_retrain',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_gen = datagen.flow_from_directory(
    './dataset_retrain',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

# Mulai fine-tuning 10 epoch
history = model.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen
)

# Simpan model baru
model.save('finetuned_best_v2.h5')
print("🎉 Model baru berhasil disimpan sebagai 'finetuned_best_v2.h5'!")